# Notebook 01 — Data Exploration
Explore chunk distribution, token stats, and document coverage after ingestion.
Run `python scripts/ingest_all.py` before opening this notebook.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config_loader import load_config
cfg = load_config()
processed_dir = Path(cfg['paths']['processed_data'])

all_chunks = []
for jf in sorted(processed_dir.glob('*.json')):
    with open(jf) as f:
        chunks = json.load(f)
    all_chunks.extend(chunks)
    print(f'{jf.name}: {len(chunks)} chunks')

print(f'\nTotal chunks: {len(all_chunks):,}')

In [ ]:
# Build a DataFrame for analysis
rows = []
for c in all_chunks:
    m = c.get('metadata', {})
    rows.append({
        'chunk_id': m.get('chunk_id', ''),
        'doc_type': m.get('doc_type', ''),
        'fiscal_period': m.get('fiscal_period', ''),
        'page_number': m.get('page_number', 0),
        'token_count': m.get('token_count', 0),
        'has_section_title': bool(m.get('section_title')),
    })

df = pd.DataFrame(rows)
print(df.describe())
print('\nChunks by doc_type:')
print(df.groupby('doc_type').size())

In [ ]:
# Token count distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['token_count'], bins=50, color='#2E86C1', edgecolor='white')
axes[0].set_xlabel('Token count per chunk')
axes[0].set_ylabel('Number of chunks')
axes[0].set_title('Token Count Distribution')
axes[0].axvline(df['token_count'].median(), color='red', linestyle='--', label=f'Median: {df["token_count"].median():.0f}')
axes[0].legend()

df.groupby('doc_type').size().plot(kind='bar', ax=axes[1], color='#1B4F72')
axes[1].set_title('Chunks by Document Type')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Flag outliers
short = df[df['token_count'] < 50]
long = df[df['token_count'] > 600]
print(f'Short chunks (< 50 tokens): {len(short)}')
print(f'Long chunks (> 600 tokens): {len(long)}')

In [ ]:
# Section title coverage
print(f'Chunks with section title detected: {df["has_section_title"].sum()} ({df["has_section_title"].mean():.1%})')

# Coverage by fiscal period
print('\nChunks by fiscal period:')
print(df.groupby('fiscal_period').size().sort_values(ascending=False))

In [ ]:
# Sample a few chunks for manual review
sample = df.sample(5)
for _, row in sample.iterrows():
    chunk = next(c for c in all_chunks if c.get('metadata', {}).get('chunk_id') == row['chunk_id'])
    print(f"\n--- {row['chunk_id']} ({row['doc_type']}, {row['fiscal_period']}, p{row['page_number']}) ---")
    print(chunk['text'][:400])
    print('...')